In [ ]:
!pip install transformers torch sacremoses -q

In [ ]:
!pip install pandas==2.2.2 -q

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("patient_text.csv")

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BioGptTokenizer,
    BioGptForCausalLM
)

# ── 1. Charger BiomedBERT ────────────────────────────────
print("Chargement BiomedBERT...")
biomedbert_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
biomedbert_tokenizer = AutoTokenizer.from_pretrained(biomedbert_name)
biomedbert_model = AutoModelForSequenceClassification.from_pretrained(
    biomedbert_name, num_labels=2
)
biomedbert_model.eval()
print("BiomedBERT chargé !")

# ── 2. Charger BioGPT ────────────────────────────────────
print("Chargement BioGPT...")
biogpt_tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")
biogpt_model = BioGptForCausalLM.from_pretrained("microsoft/biogpt")
biogpt_model.eval()
print("BioGPT chargé !")

# ── 3. Fonction classification BiomedBERT ────────────────
def classify_patient(text):
    inputs = biomedbert_tokenizer(
        text,
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = biomedbert_model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
    return "ADHD" if pred == 1 else "Control"

# ── 4. Fonction génération rapport BioGPT ────────────────
def generate_report(patient_id, tbr, faa, classification):
    report_template = f"""
CLINICAL EEG REPORT
===================
Patient ID: {patient_id}
Classification: {classification}

NEUROPHYSIOLOGICAL FINDINGS:
- Frontal Theta/Beta Ratio: {tbr:.3f}
  {"ELEVATED - indicates reduced cortical activation" if tbr > 2.5 else "NORMAL"}
- Frontal Alpha Asymmetry: {faa:.3f}
  {"ABNORMAL hemispheric asymmetry detected" if abs(faa) > 100 else "NORMAL"}

CLINICAL INTERPRETATION:
"""
    inputs = biogpt_tokenizer(
        report_template + "The neurophysiological profile indicates",
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = biogpt_model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=2.0
        )
    return biogpt_tokenizer.decode(outputs[0], skip_special_tokens=True)

# ── 5. Pipeline complet ──────────────────────────────────
def full_pipeline(patient_id, tbr, faa, text):
    print(f"\nTraitement patient {patient_id}...")

    # BiomedBERT classifie
    classification = classify_patient(text)
    print(f"BiomedBERT → {classification}")

    # BioGPT génère le rapport
    report = generate_report(patient_id, tbr, faa, classification)
    print(report)
    return report

# ── 6. Test sur un patient Nasrabadi ────────────────────
patient = df.iloc[0]

full_pipeline(
    patient_id=patient['ID'],
    tbr=3.109,
    faa=499.060,
    text=patient['text']
)

Chargement BiomedBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

BioGPT chargé !

Traitement patient v10p...
BiomedBERT → ADHD
CLINICAL EEG REPORT = = = = = = = = = = = = = = = = = = = Patient ID: v10p Classification: ADHD NEUROPHYSIOLOGICAL FINDINGS: - Frontal Theta / Beta Ratio: 3.109 ELEVATED - indicates reduced cortical activation - Frontal Alpha Asymmetry: 499.060 ABNORMAL hemispheric asymmetry detected CLINICAL INTERPRETATION: The neurophysiological profile indicates that children with ADHD are more likely to have frontal lobe dysfunction.


'CLINICAL EEG REPORT = = = = = = = = = = = = = = = = = = = Patient ID: v10p Classification: ADHD NEUROPHYSIOLOGICAL FINDINGS: - Frontal Theta / Beta Ratio: 3.109 ELEVATED - indicates reduced cortical activation - Frontal Alpha Asymmetry: 499.060 ABNORMAL hemispheric asymmetry detected CLINICAL INTERPRETATION: The neurophysiological profile indicates that children with ADHD are more likely to have frontal lobe dysfunction.'